In [ ]:
import os, json, sys
from pathlib import Path
import torch
import numpy as np
from PIL import Image
from IPython.display import display

In [ ]:
ROOT_DIR = Path.cwd().parent
print(f"Root-Verzeichnis: {ROOT_DIR}")

SRC_DIR = ROOT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from config import DATA_DIR, OUTPUT_DIR, DatasetName
from log_setup import setup_logging

setup_logging()  # INFO-Meldungen (z.B. geladene Frames) im Notebook und in logs/output.log

Root-Verzeichnis: /home/jessh/Study/Bachelor-Thesis


In [ ]:
# ---------------------------------------------------------
# Datensatz-Auswahl (gilt für alle Notebooks)
#   beide zusammen: DATASETS = [DatasetName.ANTI_UAV_410, DatasetName.CST_ANTI_UAV]
#   nur einer:      DATASETS = [DatasetName.ANTI_UAV_410]   bzw. [DatasetName.CST_ANTI_UAV]
# Kann im Notebook nach `%run 00_utils.ipynb` auch neu gesetzt werden.
# ---------------------------------------------------------
DATASETS = [DatasetName.ANTI_UAV_410, DatasetName.CST_ANTI_UAV]
print("Datensätze:", " + ".join(ds.value for ds in DATASETS))


def selections(split=None, datasets=None):
    """(Name, [Ordner]) für jeden Datensatz einzeln und danach - bei mehreren - für alle zusammen.

        for label, dirs in selections():         # Ordner = data/<Datensatz>
        for label, dirs in selections("train"):  # Ordner = data/<Datensatz>/train
    """
    datasets = DATASETS if datasets is None else list(datasets)
    dirs = [ds.root / split if split else ds.root for ds in datasets]
    for ds, d in zip(datasets, dirs):
        yield ds.value, [d]
    if len(datasets) > 1:
        yield " + ".join(ds.value for ds in datasets), dirs


def as_list(dirs):
    """Ein Ordner oder eine Liste von Ordnern -> Liste von Paths."""
    return [Path(dirs)] if isinstance(dirs, (str, Path)) else [Path(d) for d in dirs]

Erfolgreich importiert: <module 'data_handler' from '/home/jessh/Study/Bachelor-Thesis/src/data_handler/__init__.py'>


In [ ]:
def get_all_dir(root_dirs, file_endings=('.png', '.jpg', '.jpeg')):
    """Alle Bildpfade unter einem Ordner oder einer Liste von Ordnern (rekursiv, sortiert)."""
    return sorted(p for d in as_list(root_dirs) for p in d.rglob("*") if p.suffix.lower() in file_endings)

In [ ]:
def get_all_dir_names(root_dirs):
    """Alle Unterordner (rekursiv) eines Ordners oder einer Liste von Ordnern."""
    return [str(p) for d in as_list(root_dirs) for p in d.rglob("*") if p.is_dir()]

In [ ]:
def show_img(img):
    img_local = img
    # Falls es ein PyTorch-Tensor ist, in ein NumPy-Array konvertieren
    if isinstance(img, torch.Tensor):
        img_local = img.detach().cpu()
        
        # Falls Shape [C, H, W] ist (z.B. [1, H, W] oder [3, H, W])
        if img_local.ndim == 3:
            if img_local.shape[0] == 1:
                img_local = img_local.squeeze(0)  # [1, H, W] -> [H, W]
            else:
                img_local = img_local.permute(1, 2, 0)  # [3, H, W] -> [H, W, 3]
                
        # Skalierung von [0.0, 1.0] zurück auf [0, 255] und als uint8 casten
        img_local = (img_local * 255).clamp(0, 255).numpy().astype(np.uint8)
        
    display(Image.fromarray(img_local))

def show_img(img):
    """
    Robustes Anzeigen von Bildern, unabhängig davon, ob es sich um 
    einen PyTorch-Tensor oder ein NumPy-Array (Float oder uint8) handelt.
    """
    # 1. Falls es ein PyTorch-Tensor ist, auf die CPU holen und als NumPy konvertieren
    if isinstance(img, torch.Tensor):
        img = img.detach().cpu()
        # Falls Shape [C, H, W] oder [B, C, H, W] ist
        if img.ndim == 4:
            img = img[0]
        if img.ndim == 3:
            if img.shape[0] == 1:
                img = img.squeeze(0)  # [1, H, W] -> [H, W]
            else:
                img = img.permute(1, 2, 0)  # [3, H, W] -> [H, W, 3]
        img = img.numpy()
    
    # Falls das Bild im Format [C, H, W] als NumPy vorliegt (z.B. nach Tensor-Umwandlung)
    elif img.ndim == 3 and img.shape[0] in [1, 3]:
        if img.shape[0] == 1:
            img = img.squeeze(0)
        else:
            img = np.transpose(img, (1, 2, 0))

    # 2. Prüfen, ob es sich um Float-Werte handelt (z.B. im Bereich [0.0, 1.0])
    if np.issubdtype(img.dtype, np.floating):
        if img.max() <= 1.0:
            img = img * 255.0
        img = np.clip(img, 0, 255).astype(np.uint8)

    # 3. Als PIL-Image anzeigen (wird nun sauber als uint8 / Modus 'L' oder 'RGB' interpretiert)
    display(Image.fromarray(img))

In [ ]:
# ---------------------------------------------------------
# Hilfsfunktionen zur Bounding-Box / Annotationen-Suche
# ---------------------------------------------------------
def load_annotations(seq_dir):
    """
    Sucht nach Label-Dateien (JSON oder TXT/YOLO) im Sequenz-Ordner.
    Unterstützt gängige Anti-UAV Formate (IR_label.json, txt, etc.)
    """
    seq_path = Path(seq_dir)
    json_files = list(seq_path.glob("*.json"))
    txt_files = list(seq_path.glob("*.txt"))
    
    bboxes = [] # Format: [x, y, w, h] (in Pixeln)
    
    if json_files:
        for jf in json_files:
            try:
                with open(jf, 'r') as f:
                    data = json.load(f)
                    # Falls COCO / Anti-UAV JSON Struktur
                    if isinstance(data, dict):
                        if 'gt_rect' in data:
                            return data['gt_rect'] # Oft [[x,y,w,h], ...]
                        elif 'gt' in data:  # For CST-AntiUAV
                            return data['gt'] 
                        elif 'exist' in data and 'gt_rect' in data:
                            return [rect for exist, rect in zip(data['exist'], data['gt_rect']) if exist]
            except Exception:
                pass
                
    elif txt_files:
        # Falls YOLO / TXT Format vorhanden
        for tf in txt_files:
            if tf.name in ["labels.txt", "IR_label.txt"] or len(txt_files) == 1:
                try:
                    with open(tf, 'r') as f:
                        lines = f.readlines()
                        for l in lines:
                            parts = [float(x) for x in l.strip().split()]
                            if len(parts) >= 4:
                                bboxes.append(parts[-4:]) # nimmt x,y,w,h
                except Exception:
                    pass
    return bboxes
